In [ ]:
from dotenv import load_dotenv
load_dotenv()

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("olist-gold-marketing") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

df_orders = spark.read.parquet("../data/silver/orders/")
df_customers = spark.read.parquet("../data/silver/customers/")
df_items = spark.read.parquet("../data/silver/order_items/")
df_payments = spark.read.parquet("../data/silver/payments/")
df_reviews = spark.read.parquet("../data/silver/reviews/")
df_products = spark.read.parquet("../data/silver/products/")
df_sellers = spark.read.parquet("../data/silver/sellers/")
df_translation = spark.read.parquet("../data/silver/product_category_name_translation/")

In [ ]:
from pyspark.sql.functions import col

df_base = df_orders \
    .join(df_customers, "customer_id", "left") \
    .join(df_items, "order_id", "left") \
    .join(df_products, "product_id", "left") \
    .join(df_translation, "product_category_name", "left") \
    .join(df_payments, "order_id", "left") \
    .join(df_reviews, "order_id", "left") \
    .join(df_sellers, "seller_id", "left")

# Ne garder que les commandes livrées pour les analyses marketing
df_base = df_base.filter(col("order_status") == "delivered")

print("Lignes dans le DataFrame central :", df_base.count())
df_base.printSchema()

In [ ]:
from pyspark.sql.functions import avg, count, round

df_gold_satisfaction = df_reviews \
    .join(df_orders, "order_id", "left") \
    .filter(col("order_status") == "delivered") \
    .agg(
        round(avg("review_score"), 2).alias("note_moyenne"),
        count("review_id").alias("nombre_avis"),
    )

df_gold_satisfaction.show()
df_gold_satisfaction.write.mode("overwrite").parquet("../data/gold/marketing/satisfaction_globale/")

In [ ]:
from pyspark.sql.functions import avg, count, round, when, col

df_gold_retard_satisfaction = df_base \
    .groupBy("is_late") \
    .agg(
        round(avg("review_score"), 2).alias("note_moyenne"),
        count("order_id").alias("nombre_commandes")
    ) \
    .withColumn("statut_livraison",
        when(col("is_late") == 1, "En retard").otherwise("Dans les délais")
    ) \
    .select("statut_livraison", "note_moyenne", "nombre_commandes") \
    .orderBy("note_moyenne")

df_gold_retard_satisfaction.show()
df_gold_retard_satisfaction.write.mode("overwrite").parquet("../data/gold/marketing/retard_satisfaction/")

In [ ]:
from pyspark.sql.functions import sum as _sum, count, round, col

df_gold_categories = df_base \
    .groupBy("product_category_name_english") \
    .agg(
        round(_sum("price"), 2).alias("chiffre_affaires"),
        count("order_id").alias("nombre_commandes"),
        round(avg("review_score"), 2).alias("note_moyenne")
    ) \
    .filter(col("product_category_name_english").isNotNull()) \
    .orderBy(col("chiffre_affaires").desc())

df_gold_categories.show(20)
df_gold_categories.write.mode("overwrite").parquet("../data/gold/marketing/top_categories/")

In [ ]:
from pyspark.sql.functions import count, round, sum as _sum

df_gold_geo = df_base \
    .groupBy("customer_state") \
    .agg(
        count("order_id").alias("nombre_commandes"),
        round(_sum("price"), 2).alias("chiffre_affaires"),
        round(avg("review_score"), 2).alias("note_moyenne")
    ) \
    .orderBy(col("chiffre_affaires").desc())

df_gold_geo.show(30)
df_gold_geo.write.mode("overwrite").parquet("../data/gold/marketing/geo_clients/")

In [ ]:
from pyspark.sql.functions import date_format, count, round, sum as _sum

df_gold_mensuel = df_base \
    .withColumn("mois", date_format("order_purchase_timestamp", "yyyy-MM")) \
    .groupBy("mois") \
    .agg(
        count("order_id").alias("nombre_commandes"),
        round(_sum("price"), 2).alias("chiffre_affaires"),
        round(avg("review_score"), 2).alias("note_moyenne")
    ) \
    .orderBy("mois")

df_gold_mensuel.show(24)
df_gold_mensuel.write.mode("overwrite").parquet("../data/gold/marketing/evolution_mensuelle/")

In [ ]:
from pyspark.sql.functions import count, round

df_gold_fidelite = df_orders \
    .filter(col("order_status") == "delivered") \
    .groupBy("customer_id") \
    .agg(count("order_id").alias("nombre_commandes")) \
    .groupBy("nombre_commandes") \
    .agg(count("customer_id").alias("nombre_clients")) \
    .orderBy("nombre_commandes")

df_gold_fidelite.show()
df_gold_fidelite.write.mode("overwrite").parquet("../data/gold/marketing/fidelite_clients/")